# Motor de Replay de Mercado Interativo

Este notebook replica a funcionalidade do `replay_engine.py`, permitindo a execução e teste de um replay de mercado passo a passo.

**Funcionalidades:**
- Busca de dados de tick do MetaTrader 5 para um dia e ativo específicos.
- Agregação de ticks em candles de 5 minutos em tempo real simulado.
- Exibição de um gráfico de candles animado com médias móveis (SMA9, EMA21, EMA50, EMA200).

**Pré-requisito:** O terminal MetaTrader 5 deve estar aberto e logado.

### Passo 1: Configuração do Replay

**Ação:** Defina as variáveis `TICKER` e `REPLAY_DATE_STR` com os valores desejados para a simulação.

In [ ]:
# --- PARÂMETROS DE ENTRADA ---
TICKER = "WDO$"  # Ativo para o replay (deve estar no main.yaml)
REPLAY_DATE_STR = "2025-10-10" # Data no formato 'YYYY-MM-DD'
# ---------------------------

print(f"Configurado para replay do ativo '{TICKER}' na data '{REPLAY_DATE_STR}'.")

### Passo 2: Importações e Preparação do Ambiente

In [ ]:
import yaml
import logging
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import pytz
import MetaTrader5 as mt5

# Define o backend gráfico para o Matplotlib (essencial para animação fora do script)
%matplotlib tk

import mplfinance as mpf
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Valida se o ticker está configurado no main.yaml
project_root = Path.cwd().parent.parent
with open(project_root / "configs/main.yaml", "r") as file:
    config = yaml.safe_load(file)

configured_tickers = [asset['ticker'] for asset in config['assets']]
if TICKER not in configured_tickers:
    raise ValueError(f"Erro: Ticker '{TICKER}' não encontrado na lista de ativos em configs/main.yaml.")

### Passo 3: Busca dos Dados de Tick do MetaTrader 5

In [ ]:
def fetch_tick_data(ticker, replay_date_str):
    """Busca todos os dados de tick para o dia do replay."""
    replay_date = datetime.strptime(replay_date_str, "%Y-%m-%d")
    timezone = pytz.timezone("Etc/UTC")
    start_time_utc = timezone.localize(replay_date)
    end_time_utc = timezone.localize(replay_date + timedelta(days=1) - timedelta(seconds=1))
    
    logging.info(f"Conectando ao MetaTrader 5...")
    if not mt5.initialize():
        logging.error(f"Falha na inicialização do MT5: {mt5.last_error()}")
        return None

    logging.info(f"Buscando ticks para {ticker} em {replay_date.date()}...")
    ticks = mt5.copy_ticks_range(ticker, start_time_utc, end_time_utc, mt5.COPY_TICKS_ALL)
    mt5.shutdown()

    if ticks is None or len(ticks) == 0:
        logging.error(f"Nenhum dado de tick encontrado para {ticker} na data especificada.")
        return None

    ticks_df = pd.DataFrame(ticks)
    ticks_df["time"] = pd.to_datetime(ticks_df["time"], unit="s", utc=True)
    ticks_df.set_index("time", inplace=True)
    logging.info(f"{len(ticks_df)} ticks carregados com sucesso.")
    return ticks_df, start_time_utc, end_time_utc

# Executa a busca de dados
ticks_df, start_time_utc, end_time_utc = fetch_tick_data(TICKER, REPLAY_DATE_STR)

### Passo 4: Definição da Lógica de Animação e Plotagem

In [ ]:
# Variáveis globais para a animação
current_sim_time = start_time_utc

# Configuração do gráfico
chart_style = mpf.make_mpf_style(
    base_mpf_style="default",
    marketcolors=mpf.make_marketcolors(up="g", down="r", inherit=True),
    gridstyle="-",
    facecolor="white",
)

fig, ax = plt.subplots(figsize=(15, 7))
fig.suptitle(f"Replay de Mercado para {TICKER}", fontsize=16)

def update_plot(frame):
    """Função chamada a cada intervalo para atualizar o gráfico."""
    global current_sim_time # Permite modificar a variável global
    
    current_sim_time += timedelta(seconds=30)
    print(f"\rSim Time: {current_sim_time.strftime('%H:%M:%S')}", end="")

    if current_sim_time > end_time_utc:
        print("\nFim do dia de replay. Fechando o gráfico...")
        plt.close(fig)
        return

    current_ticks = ticks_df[ticks_df.index <= current_sim_time]
    if current_ticks.empty: return

    candles_m5 = current_ticks["last"].resample("5min").ohlc().dropna()
    if candles_m5.empty: return

    candles_m5["sma9"] = candles_m5["close"].rolling(window=9).mean()
    candles_m5["ema21"] = candles_m5["close"].ewm(span=21, adjust=False).mean()
    candles_m5["ema50"] = candles_m5["close"].ewm(span=50, adjust=False).mean()
    candles_m5["ema200"] = candles_m5["close"].ewm(span=200, adjust=False).mean()
    
    plot_data = candles_m5.tail(100)
    ax.clear()

    addplots = [
        mpf.make_addplot(plot_data["sma9"], color="red"),
        mpf.make_addplot(plot_data["ema21"], color="blue"),
        mpf.make_addplot(plot_data["ema50"], color="orange"),
        mpf.make_addplot(plot_data["ema200"], color="black"),
    ]

    mpf.plot(plot_data, type="candle", style=chart_style, ax=ax, addplot=addplots)
    
    last_candle = candles_m5.iloc[-1]
    ax.set_title(f"Replay de Mercado - {TICKER} (M5) - {current_sim_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\rSim Time: {current_sim_time.strftime('%H:%M:%S')} | Último Candle ({last_candle.name.strftime('%H:%M')}): O={last_candle.open:.2f} H={last_candle.high:.2f} L={last_candle.low:.2f} C={last_candle.close:.2f}", end="")


### Passo 5: Iniciar o Replay

Execute a célula abaixo para iniciar a animação. Uma nova janela com o gráfico será aberta.

In [ ]:
if ticks_df is not None:
    logging.info("Iniciando a animação do replay...")
    # Ajusta o tempo de simulação para o primeiro tick disponível
    current_sim_time = ticks_df.index[0]
    
    # Cria e exibe a animação
    ani = FuncAnimation(fig, update_plot, interval=200, save_count=1000)
    plt.show()
    print("\nReplay finalizado.")
else:
    logging.error("Não foi possível iniciar o replay pois os dados de tick não foram carregados.")